
#Can be downloaded from https://data.cityofnewyork.us/Health/NYC-Cooling-Tower-Registrations/y4fw-iqfr/about_data


In [1]:
# Import necessary libraries for data processing and spatial operations.
import pandas as pd  # For handling tabular data (CSV files).
import numpy as np  # For numerical operations and array manipulations.
from scipy.spatial import cKDTree  # For efficient spatial queries using KD-tree.
import tqdm  # Import the tqdm module for accessing its version.
from tqdm import tqdm as tqdm_progress  # Alias the tqdm function for progress bars.

# Debug: Confirm that imports are successful.
print("Debug: Libraries imported successfully.")

# Define directory paths for Kaggle environment.
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets.
sub_dir = r"/kaggle/working/"  # Submission directory for output files.

# Define file paths for input datasets.
train_file = f"{base_dir}/Training_data.csv"
valid_file = f"{base_dir}/Validation_data.csv"
tower_file = f"{base_dir}/NYC_Cooling_Tower_Registrations_20250224.csv" 
#Can be downloaded from https://data.cityofnewyork.us/Health/NYC-Cooling-Tower-Registrations/y4fw-iqfr/about_data

# Define output file paths for enriched datasets.
output_train_csv = f"{sub_dir}/training_data_COOLING_TOWER.csv"
output_valid_csv = f"{sub_dir}/validation_data_COOLING_TOWER.csv"

# Debug: Print the file paths to confirm they are set correctly.
print(f"Debug: Training file path: {train_file}")
print(f"Debug: Validation file path: {valid_file}")
print(f"Debug: Cooling tower file path: {tower_file}")
print(f"Debug: Output training CSV path: {output_train_csv}")
print(f"Debug: Output validation CSV path: {output_valid_csv}")

Debug: Libraries imported successfully.
Debug: Training file path: /kaggle/input/eyds-base-dataset/Training_data.csv
Debug: Validation file path: /kaggle/input/eyds-base-dataset/Validation_data.csv
Debug: Cooling tower file path: /kaggle/input/eyds-base-dataset/NYC_Cooling_Tower_Registrations_20250224.csv
Debug: Output training CSV path: /kaggle/working//training_data_COOLING_TOWER.csv
Debug: Output validation CSV path: /kaggle/working//validation_data_COOLING_TOWER.csv


In [2]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate the Haversine distance between two points in meters.
    Note: This function is not used in the vectorized KD-tree approach below but is included for reference.
    
    Parameters:
        lat1 (float): Latitude of the first point in degrees.
        lon1 (float): Longitude of the first point in degrees.
        lat2 (float): Latitude of the second point in degrees.
        lon2 (float): Longitude of the second point in degrees.
    
    Returns:
        float: Distance between the two points in meters.
    """
    R = 6371000  # Earth's radius in meters.
    # Convert decimal degrees to radians.
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    # Haversine formula to calculate distance.
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [3]:
def calculate_cooling_tower_features_vectorized(locations_df, tower_data, radii_meters):
    """
    Calculate cooling tower features for each location within specified radii using a vectorized KD-tree approach.
    Computes the count of active cooling towers within each radius for each location.
    
    Parameters:
        locations_df (pd.DataFrame): DataFrame containing at least 'Latitude' and 'Longitude' columns.
        tower_data (pd.DataFrame): DataFrame with cooling tower details, including 'Latitude', 'Longitude', 'Active_Equip'.
        radii_meters (list): List of radii (in meters) for which to compute the features.
    
    Returns:
        pd.DataFrame: locations_df with additional columns 'tower_count_{radius}m' for each radius.
    """
    # Debug: Print the shapes of input DataFrames.
    print(f"Debug: Locations DataFrame shape: {locations_df.shape}")
    print(f"Debug: Tower DataFrame shape: {tower_data.shape}")

    # Get location coordinates (in degrees) and convert to numpy array.
    loc_coords = locations_df[['Latitude', 'Longitude']].values

    # Debug: Print the number of locations being processed.
    print(f"Debug: Number of locations to process: {len(loc_coords)}")

    # Get tower coordinates and convert to radians for the KD-tree.
    tower_coords = tower_data[['Latitude', 'Longitude']].values
    tower_coords_rad = np.radians(tower_coords)
    tower_kdtree = cKDTree(tower_coords_rad)

    # Debug: Confirm that the KD-tree was created.
    print("Debug: KD-tree for cooling towers created successfully.")

    # Prepare dictionaries to store counts for each radius.
    result_counts = {radius: np.zeros(len(locations_df), dtype=int) for radius in radii_meters}

    # Precompute each radius in radians (Earth's radius = 6371000 meters).
    radius_radians = {radius: radius / 6371000.0 for radius in radii_meters}

    # Debug: Print the radii being used.
    print(f"Debug: Radii (in meters) for feature computation: {radii_meters}")

    # Process locations in batches for efficiency.
    batch_size = 1000
    num_batches = int(np.ceil(len(locations_df) / batch_size))

    # Debug: Print the number of batches to process.
    print(f"Debug: Number of batches to process: {num_batches}")

    for i in tqdm_progress(range(num_batches), desc="Processing locations"):
        start_idx = i * batch_size
        end_idx = min((i + 1) * batch_size, len(locations_df))
        batch_coords = loc_coords[start_idx:end_idx]
        batch_coords_rad = np.radians(batch_coords)

        # For each radius, query the KD-tree.
        for radius in radii_meters:
            r_rad = radius_radians[radius]
            # Get list of indices (for each location in the batch) of towers within the radius.
            indices_list = tower_kdtree.query_ball_point(batch_coords_rad, r=r_rad, workers=-1)
            # Loop through each location in the batch and compute features.
            for j, indices in enumerate(indices_list):
                count = len(indices)
                result_counts[radius][start_idx + j] = count

    # Append new columns to the locations DataFrame.
    for radius in radii_meters:
        col_count = f'tower_count_{radius}m'
        locations_df[col_count] = result_counts[radius]

    # Debug: Print the new columns added to the DataFrame.
    new_cols = [col for col in locations_df.columns if col.startswith('tower_count_')]
    print(f"Debug: New columns added to locations DataFrame: {new_cols}")

    return locations_df

In [4]:
def main():
    """
    Main function to calculate cooling tower features for training and validation datasets
    and save the augmented data to new CSV files.
    """
    # Define the radii (in meters) for which to calculate cooling tower features.
    radii = [100, 200, 500, 1000]

    # Debug: Print the radii being used.
    print(f"Debug: Radii for cooling tower features: {radii}")

    # Read the cooling tower data.
    print("Reading cooling tower data...")
    try:
        tower_data = pd.read_csv(tower_file)
    except FileNotFoundError:
        print(f"Error: Cooling tower data file not found at {tower_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the cooling tower data.
    print(f"Debug: Cooling tower DataFrame shape: {tower_data.shape}")
    print(f"Debug: Cooling tower DataFrame columns: {tower_data.columns.tolist()}")

    # Keep only necessary columns for cooling tower features.
    expected_cols = ['Latitude', 'Longitude', 'Active_Equip']
    tower_data = tower_data[expected_cols]

    # Debug: Confirm that columns were filtered.
    print(f"Debug: Filtered cooling tower DataFrame columns: {tower_data.columns.tolist()}")

    # Read training data.
    print("Reading training data...")
    try:
        train_data = pd.read_csv(train_file)
    except FileNotFoundError:
        print(f"Error: Training data file not found at {train_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the training data.
    print(f"Debug: Training DataFrame shape: {train_data.shape}")
    print(f"Debug: Training DataFrame columns: {train_data.columns.tolist()}")

    # Read validation data.
    print("Reading validation data...")
    try:
        validation_data = pd.read_csv(valid_file)
    except FileNotFoundError:
        print(f"Error: Validation data file not found at {valid_file}. Please check the path.")
        return

    # Debug: Print the shape and columns of the validation data.
    print(f"Debug: Validation DataFrame shape: {validation_data.shape}")
    print(f"Debug: Validation DataFrame columns: {validation_data.columns.tolist()}")

    # Ensure the training and validation datasets have 'Latitude' and 'Longitude'.
    for df, name in [(train_data, "Training"), (validation_data, "Validation")]:
        if 'Latitude' not in df.columns or 'Longitude' not in df.columns:
            print(f"Error: 'Latitude' and/or 'Longitude' columns not found in the {name} dataset.")
            return

    # Debug: Confirm that latitude and longitude columns are present.
    print("Debug: Latitude and Longitude columns verified in both datasets.")

    # Calculate cooling tower features (counts) for training data.
    print("Calculating cooling tower features for training data...")
    train_data = calculate_cooling_tower_features_vectorized(train_data.copy(), tower_data, radii)

    # Debug: Print the shape of the training data after feature calculation.
    print(f"Debug: Training DataFrame shape after feature calculation: {train_data.shape}")

    # Calculate cooling tower features for validation data.
    print("Calculating cooling tower features for validation data...")
    validation_data = calculate_cooling_tower_features_vectorized(validation_data.copy(), tower_data, radii)

    # Debug: Print the shape of the validation data after feature calculation.
    print(f"Debug: Validation DataFrame shape after feature calculation: {validation_data.shape}")

    # Drop unnecessary columns if they exist.
    cols_to_drop = ['geometry', 'datetime', 'UHI Index']
    train_data = train_data.drop(columns=cols_to_drop, errors='ignore')
    validation_data = validation_data.drop(columns=cols_to_drop, errors='ignore')

    # Debug: Print the columns after dropping.
    print(f"Debug: Training DataFrame columns after dropping: {train_data.columns.tolist()}")
    print(f"Debug: Validation DataFrame columns after dropping: {validation_data.columns.tolist()}")

    # Save the augmented data to new CSV files.
    print("Saving results...")
    train_data.to_csv(output_train_csv, index=False)
    validation_data.to_csv(output_valid_csv, index=False)

    # Debug: Print the final confirmation messages with file paths.
    print(f"Debug: Augmented training data saved to: {output_train_csv}")
    print(f"Debug: Augmented validation data saved to: {output_valid_csv}")

    print("Process completed!")

In [5]:
if __name__ == "__main__":
    main()

Debug: Radii for cooling tower features: [100, 200, 500, 1000]
Reading cooling tower data...
Debug: Cooling tower DataFrame shape: (4950, 15)
Debug: Cooling tower DataFrame columns: ['BIN', 'system_id', 'Date_Registered', 'Address', 'Borough', 'Zip_Code', 'Sample_Dates', 'Active_Equip', 'BBL', 'Latitude', 'Longitude', 'Community_Board', 'Council_District', 'Census_Tract', 'NTA_Code']
Debug: Filtered cooling tower DataFrame columns: ['Latitude', 'Longitude', 'Active_Equip']
Reading training data...
Debug: Training DataFrame shape: (11229, 4)
Debug: Training DataFrame columns: ['Longitude', 'Latitude', 'datetime', 'UHI Index']
Reading validation data...
Debug: Validation DataFrame shape: (1040, 3)
Debug: Validation DataFrame columns: ['Longitude', 'Latitude', 'UHI Index']
Debug: Latitude and Longitude columns verified in both datasets.
Calculating cooling tower features for training data...
Debug: Locations DataFrame shape: (11229, 4)
Debug: Tower DataFrame shape: (4950, 3)
Debug: Number

Processing locations: 100%|██████████| 12/12 [00:00<00:00, 49.67it/s]


Debug: New columns added to locations DataFrame: ['tower_count_100m', 'tower_count_200m', 'tower_count_500m', 'tower_count_1000m']
Debug: Training DataFrame shape after feature calculation: (11229, 8)
Calculating cooling tower features for validation data...
Debug: Locations DataFrame shape: (1040, 3)
Debug: Tower DataFrame shape: (4950, 3)
Debug: Number of locations to process: 1040
Debug: KD-tree for cooling towers created successfully.
Debug: Radii (in meters) for feature computation: [100, 200, 500, 1000]
Debug: Number of batches to process: 2


Processing locations: 100%|██████████| 2/2 [00:00<00:00, 100.92it/s]


Debug: New columns added to locations DataFrame: ['tower_count_100m', 'tower_count_200m', 'tower_count_500m', 'tower_count_1000m']
Debug: Validation DataFrame shape after feature calculation: (1040, 7)
Debug: Training DataFrame columns after dropping: ['Longitude', 'Latitude', 'tower_count_100m', 'tower_count_200m', 'tower_count_500m', 'tower_count_1000m']
Debug: Validation DataFrame columns after dropping: ['Longitude', 'Latitude', 'tower_count_100m', 'tower_count_200m', 'tower_count_500m', 'tower_count_1000m']
Saving results...
Debug: Augmented training data saved to: /kaggle/working//training_data_COOLING_TOWER.csv
Debug: Augmented validation data saved to: /kaggle/working//validation_data_COOLING_TOWER.csv
Process completed!
